In [15]:
logger = logging.getLogger(__name__)

TASK 1

In [1]:
import logging

In [16]:
import pandas as pd
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

try:
    logger.info("Loading orders dataset")

    df = pd.read_csv("C:\\Users\\Sreya\\Documents\\Tap - Python\\olist_dataset\\orders.csv")

    logger.info("Dataset loaded successfully")
    logger.info(f"Total rows: {len(df)}")

    if df.isnull().sum().any():
        logger.warning("Missing values found in dataset")

    if df.empty:
        logger.error("Dataset is empty")

except FileNotFoundError:
    logger.error("Orders file not found")

except Exception as e:
    logger.error(f"Unexpected error: {e}")

TASK 2


In [18]:
import logging
import os
from datetime import datetime

# Get current timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create logs folder
os.makedirs("logs", exist_ok=True)

# Create log filename
log_file = f"logs/validation_{timestamp}.log"

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ],
    force=True
)

# Test logging
logging.info("Data loading started")
logging.warning("Missing values found")
logging.error("Invalid records found")

print("Log file created:", log_file)

2026-11-23 11:58:51,028 - INFO - Data loading started
2026-11-23 11:58:51,030 - WARNING - Missing values found
2026-11-23 11:58:51,033 - ERROR - Invalid records found


Log file created: logs/validation_20261123_115851.log


TASK 3


In [5]:
import yaml
import pandas as pd

with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)

orders_path = config["paths"]["orders"]

min_price = config["thresholds"]["min_price"]

order_mapping = config["mappings"]["orders"]

df = pd.read_csv(r"C:\Users\Sreya\Documents\Tap - Python\olist_dataset\orders.csv")

print("Dataset loaded successfully")
print("Total rows:", len(df))
print("Minimum price:", min_price)
print("Column mappings:", order_mapping)

Dataset loaded successfully
Total rows: 500
Minimum price: 0
Column mappings: {'order_id': 'order_id', 'customer_id': 'customer_id', 'order_date': 'order_purchase_timestamp', 'status': 'order_status'}


TASK 4


In [ ]:
import pandas as pd
import logging

# Load dataset
df = pd.read_csv(r"C:\Users\Sreya\Documents\Tap - Python\olist_dataset\orders.csv")

logging.info("Dataset loaded successfully")

# Clean dataset
clean_df = df.dropna()

logging.info("Missing values removed")

# Save as CSV
clean_df.to_csv("clean_orders.csv", index=False)

logging.info("Clean CSV file saved")

# Save as JSON
clean_df.to_json("clean_orders.json", orient="records", indent=4)

logging.info("Clean JSON file saved")

print("Total rows:", len(df))
print("Clean rows:", len(clean_df))
print("Rejected rows:", len(df) - len(clean_df))

Total rows: 500
Clean rows: 458
Rejected rows: 42


TASK 5

In [9]:
import pandas as pd
import json
import time
import logging

# Start time
start_time = time.time()

# Load dataset
df = pd.read_csv(r"C:\Users\Sreya\Documents\Tap - Python\olist_dataset\orders.csv")

logging.info("Dataset loaded successfully")

# Clean dataset
clean_df = df.dropna()

# Calculate counts
total_rows = len(df)
valid_rows = len(clean_df)
rejected_rows = total_rows - valid_rows

# Calculate duration
end_time = time.time()
duration = end_time - start_time

# Create summary
summary = {
    "total_rows": total_rows,
    "valid_rows": valid_rows,
    "rejected_rows": rejected_rows,
    "duration_seconds": round(duration, 2)
}

# Save summary JSON
with open("run_summary.json", "w") as file:
    json.dump(summary, file, indent=4)

logging.info("Run summary saved successfully")

print(summary)

{'total_rows': 500, 'valid_rows': 458, 'rejected_rows': 42, 'duration_seconds': 0.01}


TASK 6


In [11]:
import os
import json
import pandas as pd

# Read existing manifest
if os.path.exists("manifest.json"):
    with open("manifest.json", "r") as file:
        manifest = json.load(file)
else:
    manifest = {}

# List of CSV files
files = [
    r"C:\Users\Sreya\Documents\Tap - Python\olist_dataset\orders.csv",
    r"C:\Users\Sreya\Documents\Tap - Python\olist_dataset\customers.csv",
    r"C:\Users\Sreya\Documents\Tap - Python\olist_dataset\products.csv"
]

# Process files
for filename in files:

    if filename in manifest:
        print(filename, "Already processed. Skipping.")

    else:
        try:
            df = pd.read_csv(filename)

            print(filename, "Processed successfully")

            manifest[filename] = "processed"

        except FileNotFoundError:
            print(filename, "File not found")

# Save updated manifest
with open("manifest.json", "w") as file:
    json.dump(manifest, file, indent=4)

print("Manifest updated successfully")

C:\Users\Sreya\Documents\Tap - Python\olist_dataset\orders.csv Processed successfully
C:\Users\Sreya\Documents\Tap - Python\olist_dataset\customers.csv Processed successfully
C:\Users\Sreya\Documents\Tap - Python\olist_dataset\products.csv Processed successfully
Manifest updated successfully


TASK 7

In [19]:
import pandas as pd
import yaml
import os
import json
import logging
import time
from datetime import datetime

# Start timer
start_time = time.time()

# Load configuration
with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)

# Create folders
os.makedirs("logs", exist_ok=True)
os.makedirs("output", exist_ok=True)

# Setup logging
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

logging.basicConfig(
    filename=f"logs/pipeline_{timestamp}.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)
logger = logging.getLogger(__name__)

# Load previous manifest
if os.path.exists("manifest.json"):
    with open("manifest.json", "r") as file:
        manifest = json.load(file)
else:
    manifest = {}

# Counters
total_rows = 0
valid_rows = 0
rejected_rows = 0

# Process each file
for name, path in config["paths"].items():

    if path in manifest:
        logging.info(f"Already processed: {path}")
        continue

    try:
        # Read CSV
        df = pd.read_csv(path)

        # Validate missing values
        clean_df = df.dropna()
        rejected_df = df[df.isnull().any(axis=1)]

        # Update counts
        total_rows += len(df)
        valid_rows += len(clean_df)
        rejected_rows += len(rejected_df)

        # Save clean CSV and JSON
        clean_df.to_csv(f"output/clean_{name}.csv", index=False)
        clean_df.to_json(
            f"output/clean_{name}.json",
            orient="records",
            indent=4
        )

        # Update manifest after successful processing
        manifest[path] = "processed"

        logger.info(f"Processed: {path}")
        logger.info(f"Valid rows: {len(clean_df)}")
        logger.warning(f"Rejected rows: {len(rejected_df)}")

    except FileNotFoundError:
        logger.error(f"File not found: {path}")

    except Exception as e:
        logger.error(f"Error processing {path}: {e}")

# Save manifest
with open("manifest.json", "w") as file:
    json.dump(manifest, file, indent=4)

# Create run summary
duration = round(time.time() - start_time, 2)

summary = {
    "total_rows": total_rows,
    "valid_rows": valid_rows,
    "rejected_rows": rejected_rows,
    "duration_seconds": duration
}

with open("output/run_summary.json", "w") as file:
    json.dump(summary, file, indent=4)

logger.info("Pipeline completed successfully!")

print("Pipeline completed!")
print(summary)

Pipeline completed!
{'total_rows': 0, 'valid_rows': 0, 'rejected_rows': 0, 'duration_seconds': 0.01}


TASK 8

In [13]:
!pip install ruff

   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/10.4 MB 8.7 MB/s eta 0:00:02
   ---------------- ----------------------- 4.2/10.4 MB 11.8 MB/s eta 0:00:01
   --------------------------- ------------ 7.1/10.4 MB 12.4 MB/s eta 0:00:01
   ------------------------------------- -- 9.7/10.4 MB 12.4 MB/s eta 0:00:01
   ---------------------------------------- 10.4/10.4 MB 11.9 MB/s  0:00:00


In [20]:
!ruff check validator.py transformer.py test_functions.py reader.py pytest.py

All checks passed!
